In [1]:
import pandas as pd
import json

from pymongo import MongoClient

# Función genérica para cargar JSON

In [2]:
def cargar_json(ruta) -> pd.DataFrame:
    with open(ruta, "r", encoding="utf-8") as f:
        data = json.load(f)

    return pd.DataFrame(data)

In [3]:
df_locales = cargar_json("../doc_files/json_files/locales202312.json")
df_licencias = cargar_json("../doc_files/json_files/licencias202312.json")
df_terrazas = cargar_json("../doc_files/json_files/terrazas202312.json")
df_actividadeconomica = cargar_json("../doc_files/json_files/actividadeconomica202312.json")

# 1.2. Implementación del modelo

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["censo_locales_db"]
collection = db["locales"]

# ==========================
# AGRUPAR RELACIONES
# ==========================

licencias_dict = (
    df_licencias.groupby("id_local")
    .apply(lambda x: x.drop(columns=["id_local"], errors="ignore").to_dict("records"))
    .to_dict()
)

terrazas_dict = (
    df_terrazas.groupby("id_local")
    .apply(lambda x: x.drop(columns=["id_local"], errors="ignore").to_dict("records"))
    .to_dict()
)

actividadeconomica_dict = (
    df_actividadeconomica.groupby("id_local")
    .apply(lambda x: x.drop(columns=["id_local"], errors="ignore").to_dict("records"))
    .to_dict()
)

# ===============================
# CONSTRUIR Y CARGAR DOCUMENTOS
# ===============================

documentos = []

for row in df_locales.to_dict("records"):

    id_local = row["id_local"]

    doc = {
        "_id": int(id_local),
        "local": row,
        "licencias": licencias_dict.get(id_local, []),
        "terrazas": terrazas_dict.get(id_local, []),
        "actividadeconomica": actividadeconomica_dict.get(id_local, [])
    }

    documentos.append(doc)

print("Documentos:", len(documentos))

collection.insert_many(documentos, ordered=False)


Documentos: 151162


InsertManyResult([20000596, 20000605, 20000669, 20000709, 20000721, 20000729, 20000756, 20000761, 20000764, 20000766, 20000783, 20000797, 20000811, 20000813, 20000864, 20000880, 20000899, 20000902, 20000903, 20000920, 20000937, 20000939, 20000948, 20000956, 20000958, 20000959, 20001008, 20001013, 20001015, 20001003, 20001010, 20001027, 20001029, 20001031, 20001049, 20001053, 20001057, 20001080, 20001084, 20001085, 20001088, 20000767, 20000793, 20000795, 20000809, 20000814, 20000818, 20000822, 20000824, 20000826, 20000829, 20000844, 20000852, 20000887, 10000004, 10000116, 10000150, 10000224, 10000264, 10000412, 10000413, 10000442, 10000455, 10000462, 20000904, 20000926, 20000944, 20000946, 20000965, 20000974, 20000985, 20000990, 20000998, 20001016, 10000463, 10000478, 10000003, 10000044, 10000097, 10000102, 10000162, 10000385, 10000398, 10000401, 20001028, 20001102, 20001104, 20001107, 20001109, 20001124, 20001127, 20001159, 20001179, 20001186, 10000422, 10000451, 10000503, 10000534, 10

In [ ]:
collection.count_documents({})


In [ ]:
doc = collection.find_one()
for doc in collection.find({}, {"_id": 0, "id_local": 1}).limit(5):
    print(doc)



{}
{}
{}
{}
{}


Insertar en MongoDB